In [ ]:
!nvidia-smi

In [ ]:
MODEL_NAME = "microsoft/Phi-4-mini-instruct"
DATASET_PATH = "loknezmonzter/pmc-patients-distilled-medgemma-22B"
OUTPUT_DIR = "checkpoints/phi-4-mini-instruct-ft"
MAX_SEQ_LENGTH = 4096
# NUM_TRAIN_EPOCHS = 3 # For dry run prefer using steps
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 7e-5
SEED = 3407

In [ ]:
import json

JSON_SCHEMA = {
    "summary": "A concise, 1-2 sentence abstractive summary of the clinical scenario.",
    "clinical_reasoning": "A step-by-step logical breakdown of the diagnoses, treatments, or clinical decisions made in the text. Explain WHY certain relationships exist. Keep short and brief but to the point.",
    "relationships": [
        {
            "subject": "Source entity (e.g., Patient, Drug, Symptom)",
            "predicate": "Use STANDARD POSITIVE relationships (e.g., HAS_HISTORY, SHOWS_SYMPTOM, DIAGNOSED_WITH, PRESCRIBED). Do not use negated verbs like 'DENIES' or 'LACKS'.",
            "object": "Target entity",
            "polarity": "positive OR negative (Use 'negative' if the patient denies the history or lacks the symptom)",
            "certainty": "confirmed, suspected, OR hedged",
            "evidence": "The exact verbatim text snippet that proves this relationship."
        }
    ],
    "keywords": ["List", "of", "important", "clinical", "NER", "terms"]
}
SCHEMA_STRING = json.dumps(JSON_SCHEMA, indent=4)

# Finalized system prompt
SYSTEM_PROMPT = (
    "You are an expert clinical informatician. "
    f"Extract data strictly into this JSON schema:\n\n{SCHEMA_STRING}\n\n"
    "CRITICAL RULES:\n"
    "1. Use ONLY double quotes for all JSON keys and string values.\n"
    "2. Response MUST start with '{' and end with '}'.\n"
    "3. Output raw JSON only — no markdown, no code blocks.\n"
    "4. Provide values for ALL keys in the schema.\n"
    "5. Apostrophes in clinical terms (e.g., patient's) are allowed inside double-quoted strings.\n"
    "6. Extract at max 10 most clinically significant relationships only. "
    "Prioritize: diagnosis > treatment > symptoms > history.\n"
)

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Load and configure tokenizer padding
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Create bits and bytes configuration for n-bit fine tuning
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Load the model with bits and bytes config
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="flash_attention_2",
    dtype=torch.bfloat16
)

# Required for 4-bit training
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
model.config.pretraining_tp = 1 # keep 1 for single GPU training (no distribution)

# LoRA configuration
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "qkv_proj",      # Fused Q+K+V (replaces q_proj, k_proj, v_proj)
        "o_proj",        # Attention output 
        "gate_up_proj",  # Fused gate+up MLP (replaces gate_proj, up_proj)
        "down_proj",     # MLP down
    ],
)

model = get_peft_model(model, peft_config=lora_config)
model.print_trainable_parameters()

for name, param in model.named_parameters():
    if param.dtype in (torch.float32, torch.float16):
        param.data = param.data.to(torch.bfloat16)

print("✅ All non-quantized params cast to bfloat16")